# E4 · Ablación de LoRA y proyección visual

**Estado:** protocolo prospectivo, fijado antes de observar nuevos tests.

**Objetivo y pregunta.** Distinguir el efecto de LoRA y de la proyección visual entrenada en el E4 real. ¿Qué conserva o mejora cada componente por separado?

**Hipótesis.** La adaptación combinada puede aportar señales complementarias, pero esa contribución debe contrastarse con cada componente aislado.

**Intervención.** Solo LoRA q/v o solo proyección visual; semillas 45, 46 y 47, siempre P8×K4. El encoder se construye del mismo CLIP fijado, con adaptadores inicialmente nulos. En solo proyección, toda la visión (incluidos los adaptadores nulos) queda congelada; en solo LoRA, se congela la proyección.

**Controles.** Las tres corridas P8×K4 combinadas de `e4_sampling_multiseed.ipynb` sirven como control contemporáneo. Nunca se utiliza P16×K2 para atribuir efectos a los componentes. Cabeza E3 seed 46 / versión 20 congelada; mismos datos, recorte, margen, tasa, máximo de pasos y paciencia. El histórico v23 es otra referencia, cuya inicialización no estaba sembrada antes de construir LoRA.

**Muestra y protocolo.** Train/val públicos refinados v2, separados por identidad; el test público v1 permanece inmutable (v2 añade cajas). Evaluación de 277 fotos públicas y 502 fotos privadas, con galerías separadas. No se entrena con PetTrace ni con identidades de test.

**Selección.** Checkpoint por mAP macro de validación; componente por media de tres semillas, empate favorece solo LoRA; semilla por validación, empate por semilla menor. Se congela la decisión antes de evaluar los tests. Todas las condiciones se publican, incluidas las negativas. La comparación entre tres brazos informa atribución y no presupone superioridad de ninguno.

**Métricas.** Rank-1/2/5/10, mAP, errores por consulta/especie y dispersión descriptiva entre semillas. La EER es descriptiva sobre los mismos pares. Los resultados se completan solo después de publicación verificable.

**Limitaciones.** Tres semillas no estiman incertidumbre poblacional. Benchmark ya observado, sin separación entre sesiones garantizada; recuperación cerrada de la misma especie, leave-one-out por fotografía. Las tasas y el presupuesto se mantienen iguales para aislar la intervención; no son una búsqueda del óptimo de cada componente.

LoRA: [Hu et al.](https://arxiv.org/abs/2106.09685). Batch-hard: [Hermans et al.](https://arxiv.org/abs/1703.07737). CLIP: [Radford et al.](https://proceedings.mlr.press/v139/radford21a.html). YOLOv8: [Ultralytics](https://docs.ultralytics.com/models/yolov8/). Brazos, muestreo y selección son decisiones propias.

### Entorno
Mismo paquete publicado 0.2.46. La semilla se fija antes de construir los adaptadores. Publicación obligatoria; registro activado solo para el seleccionado, sin promover aliases.

In [ ]:
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks,tracking]==0.2.46"

In [ ]:
import json, os, sys
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import numpy as np
import torch
from IPython.display import Markdown, display
from google.colab import userdata
import pettrace
from pettrace.tracking import PRODUCTION_URI, source_identity

if not (3, 11) <= sys.version_info[:2] < (3, 14) or sys.platform != "linux":
    raise RuntimeError("E4 requires Colab Linux with Python 3.11–3.13")
if pettrace.__version__ != "0.2.46":
    raise RuntimeError("Install PetTrace 0.2.46 before continuing")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; a T4 is sufficient")
print("Access [1/2] Loading Colab Secrets…")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise TypeError(f"Colab Secret {name} must be a non-empty text value")
    os.environ[name] = value
print("Access [2/2] Credentials available.")
EXPERIMENT, DATASET = "E4", "v6"
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
TRAINING_DATASET_VERSION, SOURCE_SAMPLE_PCT = "v2", 100
SEEDS, ARMS = (45, 46, 47), ("lora_only", "projection_only")
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
BASE_MODEL = "models:/pettrace-embedder/2"
LOG_TO_MLFLOW = True
REGISTER_E4 = True
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
REPORT = Path("/content/reports") / "e4_component_study" / RUN_LABEL
REPORT.mkdir(parents=True)
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
print("Package:", pettrace.__version__, "| GPU:", torch.cuda.get_device_name(0), "| Report:", REPORT)
STUDY = {"id": RUN_LABEL, "seeds": list(SEEDS), "arms": list(ARMS), "batch": [8, 4],
         "control_study": "20261118T053018Z-9d7ddbb5", "control_arm": "P8 K4 LoRA plus projection",
         "initialization_seeded": True, "selection": "mean validation mAP per arm, then best validation seed",
         "tie_break": "lora_only, then lower seed", "test_used_for_selection": False}
(REPORT / "study-protocol.json").write_text(json.dumps(STUDY, indent=2))

## Fase 1. Preparación y preprocesamiento de datos
Se cargan snapshots ya preparados; no se limpia ni redivide el corpus. Las cajas YOLO se verifican contra el detector E2. Las fotografías privadas solo se utilizan para evaluación separada.

In [ ]:
from pettrace.data import training
from pettrace.data.volume import training_volume

print("Dataset [1/2] Downloading pinned public v2 shards and verifying hashes…")
prepared = training(version=TRAINING_DATASET_VERSION,
    cache_dir="/content/datasets/pettrace-refined", token=os.environ["HF_TOKEN"])
manifest_rows = prepared.rows
training_source = prepared.provenance
coverage_metrics, coverage_params = training_volume(prepared, SOURCE_SAMPLE_PCT, REPORT)
(REPORT / "transport-equivalence.json").write_text(json.dumps(training_source, indent=2))
print("Dataset [2/2] Verified:", prepared.fingerprint, "| photos:", len(manifest_rows),
      "| identities:", prepared.summary["retained_identities"],
      "| commits:", [source.commit[:12] for source in prepared.sources])

In [ ]:
from pettrace.tracking import production_context
from pettrace.experiments.e3 import load_benchmarks
production_context()
benchmarks = load_benchmarks(prepared, REPORT, private_version=DATASET, private_commit=DATASET_COMMIT)
private_ds, public_ds, PUBLIC_REPORT = benchmarks.private, benchmarks.public, benchmarks.public_report
print("Public test:", len(public_ds.photos), "photos | PetTrace:", len(private_ds.photos), "photos")

## Fase 2. Representación y entrenamiento
Se aísla cada componente y se verifica el conjunto exacto de pesos entrenables antes de ajustar. Cada corrida conserva configuración, lista de parámetros, paridad, checkpoint, curvas y tiempos. Las seis se entrenan antes de observar métricas de test. Un resumen existente permite retomar la exportación sin reentrenar; un fallo deja los archivos locales disponibles.

In [ ]:
from pettrace.evaluation.core import load_model
from pettrace.experiments.e4 import load_control_head
from pettrace.training.lora import Config, PreparedPreprocessor, build_encoder, control_equivalence

base_model = load_model(BASE_MODEL)
if not base_model.golden_ok or base_model.pipeline.detector is None:
    raise RuntimeError("The fixed E2 base failed verification")

head, control_receipt = load_control_head(REPORT)
preprocessor = PreparedPreprocessor(base_model.pipeline, prepared)

training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
train_rows = [row for row in training_rows if row["split"] == "train"]
gate = [
    next(row for row in sorted(train_rows, key=lambda item: item["sha256"])
         if row["species"] == species)
    for species in ("cat", "dog")
]

checked = preprocessor.validate(gate)
pixels = preprocessor.pixels(gate)
e2 = base_model.pipeline.session.run(
    [base_model.pipeline.output_name],
    {base_model.pipeline.input_name: pixels},
)[0]

In [ ]:
import copy, gc, shutil, time, yaml
from pettrace.training.triplet import seed_all
from pettrace.training.lora import fit, trainable_parameters, export_encoder, load_adapter
from pettrace.experiments.e3 import select_validation_candidate

candidates = []
p, k = 8, 4
for arm in ARMS:
    for seed in SEEDS:
        run_report = REPORT / f"{arm}-seed{seed}"
        run_report.mkdir(exist_ok=True)
        for filename in ("dataset-coverage.json", "sources.json", "transport-equivalence.json", "study-protocol.json"):
            shutil.copy2(REPORT / filename, run_report / filename)
        config = Config(seed=seed, identities_per_batch=p, images_per_identity=k)
        seed_all(seed)
        encoder, adapted_layers = build_encoder(copy.deepcopy(head), CLIP_REVISION, config=config)
        if arm == "lora_only":
            encoder.visual_projection.requires_grad_(False)
        else:
            encoder.vision_model.requires_grad_(False)
        parity = control_equivalence(encoder, pixels, e2)
        trainables = {name: list(value.shape) for name, value in encoder.named_parameters() if value.requires_grad}
        if arm == "lora_only":
            assert len(trainables) == 48 and all(name.endswith((".a", ".b")) for name in trainables)
        else:
            assert set(trainables) == {"visual_projection.weight"}
        run_config = {"experiment": EXPERIMENT, "arm": arm, "training": asdict(config), "study": STUDY,
                      "control": control_receipt, "clip_revision": CLIP_REVISION,
                      "source": SOURCE_IDENTITY, "trainable_parameters": trainable_parameters(encoder),
                      "trainable_names_and_shapes": trainables, "log_to_mlflow": True,
                      "register_selected_model": REGISTER_E4}
        (run_report / "training-config.json").write_text(json.dumps(run_config, indent=2))
        (run_report / "control-equivalence.json").write_text(json.dumps(parity, indent=2))
        summary_path = run_report / "training-summary.json"
        if summary_path.exists():
            training_summary = json.loads(summary_path.read_text())
            assert training_summary["config"] == asdict(config), "Stored configuration differs"
            load_adapter(encoder, run_report / "adapter.pt")
        else:
            started = time.monotonic()
            print(f"Training {arm}, P={p}, K={k}, seed={seed}; validation only", flush=True)
            encoder, training_summary = fit(encoder, training_rows, preprocessor, run_report,
                                            config=config, device="cuda")
            training_summary["elapsed_seconds"] = time.monotonic() - started
            summary_path.write_text(json.dumps(training_summary, indent=2))
        model_dir = run_report / "model"
        if not (model_dir / "golden.json").exists():
            export_encoder(base_model.dir, encoder, model_dir,
                model_version=f"clip-vit-b32-crop-{arm}-p8k4-seed{seed}@study1",
                verification_pixels=pixels)
            spec = yaml.safe_load((model_dir / "pipeline.yaml").read_text())
            spec["adaptation"].update(method=arm,
                visual_projection_trainable=arm == "projection_only",
                targets=["q_proj", "v_proj"] if arm == "lora_only" else [],
                base="frozen CLIP vision base and frozen E3-100 head")
            (model_dir / "pipeline.yaml").write_text(yaml.safe_dump(spec, sort_keys=False))
        candidates.append({"arm": arm, "p": p, "k": k, "seed": seed, "report": run_report,
            "model_dir": model_dir, "config": config, "training_summary": training_summary,
            "best_val_mAP": training_summary["best_val_mAP"], "best_epoch": training_summary["best_epoch"]})
        del encoder
        gc.collect(); torch.cuda.empty_cache()
        print(f"Finished training {arm}, seed={seed}: validation mAP={training_summary['best_val_mAP']:.6f}", flush=True)

assert len(candidates) == 6
family_scores = {arm: float(np.mean([c["best_val_mAP"] for c in candidates if c["arm"] == arm])) for arm in ARMS}
selected_arm = max(ARMS, key=lambda arm: (family_scores[arm], -ARMS.index(arm)))
selected = select_validation_candidate([c for c in candidates if c["arm"] == selected_arm])
selection = {"selected_before_test_evaluation": True, "family_validation_means": family_scores,
             "selected_arm": selected_arm, "selected_seed": selected["seed"], "study": STUDY,
             "candidates": [{key: c[key] for key in ("arm", "p", "k", "seed", "best_epoch", "best_val_mAP")} for c in candidates]}
(REPORT / "selection.json").write_text(json.dumps(selection, indent=2))
for candidate in candidates:
    shutil.copy2(REPORT / "selection.json", candidate["report"] / "selection.json")
print("Selection frozen before test:", selection)

## Fase 3. Evaluación y análisis
Se evalúan los seis candidatos sin modificar la selección ya guardada. Rank-k agrega el máximo por identidad; AP/mAP ordena fotografías. Se publican todos los resultados en MLflow antes de registrar el seleccionado. Si falla esta celda, se reejecuta sin repetir la fase de entrenamiento; los recibos existentes evitan duplicar evaluaciones terminadas en esta sesión. Los resultados y limitaciones se completan con la evidencia real, sin asegurar mejoras de antemano.

In [ ]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save
from pettrace.experiments.e3 import publish_results, register_candidate
from pettrace.tracking import save_provenance

for candidate in candidates:
    run_report = candidate["report"]
    run_public_report = run_report / "public_test"
    run_public_report.mkdir(exist_ok=True)
    shutil.copy2(PUBLIC_REPORT / "dataset-selection.json", run_public_report / "dataset-selection.json")
    e4_model = load_model(str(candidate["model_dir"]))
    if not e4_model.golden_ok:
        raise RuntimeError("Exported E4 failed golden verification")
    receipt = run_report / "publication.json"
    if receipt.exists() and json.loads(receipt.read_text()).get("published"):
        publication = json.loads(receipt.read_text())
        client = mlflow.MlflowClient()
        assert all(client.get_run(publication[key]).info.status == "FINISHED" for key in ("run_id", "public_run_id"))
    else:
        public_ev = evaluate(public_ds, e4_model.pipeline, zero_shot=True)
        private_ev = evaluate(private_ds, e4_model.pipeline, zero_shot=True)
        save(public_ev, e4_model, str(run_public_report))
        save(private_ev, e4_model, str(run_report))
        save_provenance(run_public_report, e4_model)
        publication = publish_results(public_ev, private_ev, e4_model, run_report, run_public_report, SOURCE_SHA256,
            experiment=EXPERIMENT, source_sample_pct=SOURCE_SAMPLE_PCT, config=candidate["config"],
            training_summary=candidate["training_summary"], prepared_fingerprint=prepared.fingerprint,
            training_source=training_source, coverage_metrics=coverage_metrics,
            coverage_params=coverage_params, private_dataset_version=DATASET)
    candidate["publication"] = publication
    for run_id in (publication["run_id"], publication["public_run_id"]):
        mlflow.MlflowClient().set_tag(run_id, "study_id", RUN_LABEL)
        mlflow.MlflowClient().set_tag(run_id, "adaptation", candidate["arm"])
    print("Published:", candidate["arm"], candidate["seed"], publication, flush=True)
    del e4_model
    gc.collect()

if REGISTER_E4:
    publication = selected["publication"]
    registration = register_candidate(selected["report"], publication["run_id"], publication["public_run_id"], experiment=EXPERIMENT)
    print("Selected candidate version:", registration["version"], "| no alias promoted")

In [ ]:
results = []
for candidate in candidates:
    for domain, folder in (("public", candidate["report"] / "public_test"), ("private", candidate["report"])):
        summary = json.loads((folder / "summary.json").read_text())
        results.append({"domain": domain, "arm": candidate["arm"], "p": candidate["p"], "k": candidate["k"], "seed": candidate["seed"],
                        "best_val_mAP": candidate["best_val_mAP"], "metrics": summary["global"],
                        "publication": candidate["publication"]})
(REPORT / "study-results.json").write_text(json.dumps(results, indent=2))
for domain in ("public", "private"):
    for arm in ARMS:
        group = [r for r in results if r["domain"] == domain and r["arm"] == arm]
        print(domain, arm, {metric: {"mean": float(np.mean([r['metrics'][metric] for r in group])),
              "sd_seeds": float(np.std([r['metrics'][metric] for r in group], ddof=1))} for metric in ("rank1", "rank2", "mAP")})
for run_id in (selected["publication"]["run_id"], selected["publication"]["public_run_id"]):
    mlflow.MlflowClient().log_artifact(run_id, str(REPORT / "study-results.json"), artifact_path="study")
    mlflow.MlflowClient().log_artifact(run_id, str(REPORT / "selection.json"), artifact_path="study")
print("Study complete: six candidates, both domains published. Selected only by validation.")